# Page and Section Index Retrieval [Step 1 - Metadata as Index]

> **MLCourse - Agentic AI - Vectorless RAG**

This notebook demonstrates how to build a retrieval system that uses
page numbers and section headings as the primary index. No embeddings
or vectors are involved. We extract text from the Transformer paper
PDF, attach page and section metadata to every chunk, and retrieve
passages by navigating the document structure directly.

In [1]:
# Import all libraries needed for this notebook.
import re                              # Regex for section detection
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path               # File path handling
from pypdf import PdfReader            # PDF text extraction

In [2]:
# ## Part 1: Configuration
# Define paths and constants for the entire notebook.

PDF_PATH = r"D:\projects\python\MLCourse\03_agentic_ai\data\attention_is_all_you_need.pdf"
CHUNK_SIZE = 600                       # Target chunk size in characters
OVERLAP = 100                          # Overlap between adjacent chunks

print(f"PDF path: {PDF_PATH}")
print(f"Chunk size: {CHUNK_SIZE}, overlap: {OVERLAP}")

PDF path: D:\projects\python\MLCourse\03_agentic_ai\data\attention_is_all_you_need.pdf
Chunk size: 600, overlap: 100


In [3]:
# ## Part 2: Extracting Text and Page Metadata
# We read every page from the PDF and store the raw text along with
# the page number. Page numbers are our first metadata field.

reader = PdfReader(PDF_PATH)
total_pages = len(reader.pages)
print(f"PDF has {total_pages} pages")

# Extract text from each page, stripping Unicode artifacts.
pages_raw = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    # Normalize common Unicode issues in academic PDFs.
    text = text.replace("\u2217", "*")     # asterisk-like char
    text = text.replace("\u2019", "'")     # right single quote
    text = text.replace("\u2013", "-")     # en dash
    text = text.replace("\u0142", "l")     # Polish l with stroke
    text = text.replace("\u0105", "a")     # Polish a with ogonek
    pages_raw.append({"page": i, "text": text})
    preview = text[:80].replace("\n", " ")
    print(f"  Page {i}: {len(text)} chars -- {preview}...")

PDF has 15 pages
  Page 0: 2857 chars -- Provided proper attribution is provided, Google hereby grants permission to repr...
  Page 1: 4251 chars -- 1 Introduction Recurrent neural networks, long short-term memory [13] and gated ...
  Page 2: 1823 chars -- Figure 1: The Transformer - model architecture. The Transformer follows this ove...
  Page 3: 2491 chars -- Scaled Dot-Product Attention  Multi-Head Attention Figure 2: (left) Scaled Dot-P...


  Page 4: 3181 chars -- output values. These are concatenated and once again projected, resulting in the...
  Page 5: 3450 chars -- Table 1: Maximum path lengths, per-layer complexity and minimum number of sequen...
  Page 6: 3300 chars -- length n is smaller than the representation dimensionality d, which is most ofte...


  Page 7: 3178 chars -- Table 2: The Transformer achieves better BLEU scores than previous state-of-the-...
  Page 8: 2973 chars -- Table 3: Variations on the Transformer architecture. Unlisted values are identic...
  Page 9: 3111 chars -- Table 4: The Transformer generalizes well to English constituency parsing (Resul...
  Page 10: 3216 chars -- [5] Kyunghyun Cho, Bart van Merrienboer, Caglar Gulcehre, Fethi Bougares, Holger...
  Page 11: 3220 chars -- [25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini. Building...


  Page 12: 812 chars -- Attention Visualizations Input-Input Layer5 It is in this spirit that a majority...


  Page 13: 815 chars -- Input-Input Layer5 The Law will never be perfect , but its application should be...


  Page 14: 818 chars -- Input-Input Layer5 The Law will never be perfect , but its application should be...


In [4]:
# ## Part 3: Detecting Section Headings
# Academic papers follow a predictable section structure. We scan for
# numbered headings and title-like lines to build a section map.

# Common section headings in the Transformer paper.
SECTION_PATTERNS = [
    r"^(?:Abstract)\b",
    r"^\d+\s+Introduction\b",
    r"^\d+\s+Background\b",
    r"^\d+\s+Model Architecture\b",
    r"^\d+\s+Why Self-Attention\b",
    r"^\d+\s+Training\b",
    r"^\d+\s+Results\b",
    r"^\d+\s+Conclusion\b",
    r"^Attention Is All You Need\b",      # Title line
]

section_map = []  # List of (page_index, section_name) tuples.

for page_info in pages_raw:
    page_idx = page_info["page"]
    lines = page_info["text"].split("\n")
    for line in lines:
        line_stripped = line.strip()
        for pattern in SECTION_PATTERNS:
            if re.match(pattern, line_stripped, re.IGNORECASE):
                section_map.append((page_idx, line_stripped))

print("Detected sections:")
for pg, sec in section_map:
    print(f"  Page {pg}: {sec}")

Detected sections:
  Page 0: Attention Is All You Need
  Page 0: Abstract
  Page 1: 1 Introduction
  Page 1: 2 Background
  Page 1: 3 Model Architecture
  Page 5: 4 Why Self-Attention
  Page 6: 5 Training
  Page 7: 6 Results
  Page 9: 7 Conclusion


In [5]:
# ## Part 4: Assigning Sections to Pages
# Each page belongs to the most recently detected section. We build a
# lookup that maps every page index to its section name.

def build_page_to_section(section_map, total_pages):
    """Map each page to the section it belongs to.

    Walks through detected sections and assigns all pages between
    two consecutive headings to the earlier heading.
    """
    page_section = {}
    for pg, sec in section_map:
        page_section[pg] = sec

    # Forward-fill: each page inherits the section of the nearest
    # previous page that has a detected heading.
    current_section = "Title"
    result = {}
    for pg in range(total_pages):
        if pg in page_section:
            current_section = page_section[pg]
        result[pg] = current_section
    return result

page_to_section = build_page_to_section(section_map, total_pages)

print("Page-to-section mapping:")
for pg in range(total_pages):
    print(f"  Page {pg} -> {page_to_section[pg]}")

Page-to-section mapping:
  Page 0 -> Abstract
  Page 1 -> 3 Model Architecture
  Page 2 -> 3 Model Architecture
  Page 3 -> 3 Model Architecture
  Page 4 -> 3 Model Architecture
  Page 5 -> 4 Why Self-Attention
  Page 6 -> 5 Training
  Page 7 -> 6 Results
  Page 8 -> 6 Results
  Page 9 -> 7 Conclusion
  Page 10 -> 7 Conclusion
  Page 11 -> 7 Conclusion
  Page 12 -> 7 Conclusion
  Page 13 -> 7 Conclusion
  Page 14 -> 7 Conclusion


In [6]:
# ## Part 5: Chunking with Metadata
# We split each page into overlapping chunks and attach metadata
# (page number, section name) to every chunk. This metadata is the
# backbone of our vectorless retrieval system.

chunks = []
for page_info in pages_raw:
    page_idx = page_info["page"]
    text = page_info["text"]
    section = page_to_section[page_idx]

    # Split the page text into overlapping windows.
    start = 0
    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))
        chunk_text = text[start:end]
        chunks.append({
            "text": chunk_text,
            "page": page_idx,
            "section": section,
            "chunk_id": len(chunks),
            "char_start": start,
            "char_end": end,
        })
        start += CHUNK_SIZE - OVERLAP

print(f"Created {len(chunks)} chunks across {total_pages} pages")
print(f"Avg chunk length: {sum(len(c['text']) for c in chunks) // len(chunks)} chars")

Created 85 chunks across 15 pages
Avg chunk length: 547 chars


In [7]:
# ## Part 6: Retrieval by Page Number
# The simplest form of vectorless retrieval: find all chunks on a
# specific page or range of pages.

def retrieve_by_page(target_page, chunk_list):
    """Return all chunks from the given page number."""
    return [c for c in chunk_list if c["page"] == target_page]

def retrieve_by_page_range(start_page, end_page, chunk_list):
    """Return all chunks within a page range (inclusive)."""
    return [c for c in chunk_list if start_page <= c["page"] <= end_page]

# Test: retrieve all content from page 2.
results = retrieve_by_page(2, chunks)
print(f"Retrieved {len(results)} chunks from page 2")
for c in results:
    preview = c["text"][:60].replace("\n", " ")
    print(f"  [{c['section']}] {preview}...")

Retrieved 4 chunks from page 2
  [3 Model Architecture] Figure 1: The Transformer - model architecture. The Transfor...
  [3 Model Architecture] ected feed-forward network. We employ a residual connection ...
  [3 Model Architecture] n addition to the two sub-layers in each encoder layer, the ...
  [3 Model Architecture] , ensures that the predictions for positioni can depend only...


In [8]:
# ## Part 7: Retrieval by Section Name
# Retrieve all chunks belonging to a named section. This is useful
# when you know which part of the paper contains the answer.

def retrieve_by_section(section_name, chunk_list):
    """Return all chunks whose section matches the given name (case-insensitive)."""
    target = section_name.lower()
    return [c for c in chunk_list if target in c["section"].lower()]

# Test: retrieve all content from the Training section.
results = retrieve_by_section("Training", chunks)
print(f"Retrieved {len(results)} chunks from 'Training' section")
for c in results[:3]:
    preview = c["text"][:60].replace("\n", " ")
    print(f"  [Page {c['page']}] {preview}...")
if len(results) > 3:
    print(f"  ... and {len(results) - 3} more chunks")

Retrieved 7 chunks from 'Training' section
  [Page 6] length n is smaller than the representation dimensionality d...
  [Page 6] h length toO(n/r). We plan to investigate this approach furt...
  [Page 6] ctor of k. Separable convolutions [ 6], however, decrease th...
  ... and 4 more chunks


In [9]:
# ## Part 8: Keyword Search Within a Section
# Combine structural navigation with keyword filtering for precise
# retrieval. We search for a keyword only within a specific section.

def retrieve_by_keyword_in_section(keyword, section_name, chunk_list):
    """Keyword search restricted to a given section."""
    keyword_lower = keyword.lower()
    section_chunks = retrieve_by_section(section_name, chunk_list)
    matches = []
    for c in section_chunks:
        if keyword_lower in c["text"].lower():
            matches.append(c)
    return matches

# Test: find mentions of "attention" in the Model Architecture section.
results = retrieve_by_keyword_in_section("attention", "Model Architecture", chunks)
print(f"Found {len(results)} chunks mentioning 'attention' in Model Architecture")
for c in results[:3]:
    # Show a snippet around the keyword.
    idx = c["text"].lower().find("attention")
    start = max(0, idx - 30)
    end = min(len(c["text"]), idx + 50)
    snippet = c["text"][start:end].replace("\n", " ")
    print(f"  [Page {c['page']}] ...{snippet}...")

Found 18 chunks mentioning 'attention' in Model Architecture
  [Page 1] ...omputation, however, remains. Attention mechanisms have become an integral part ...
  [Page 1] ...nstead relying entirely on an attention mechanism to draw global dependencies be...
  [Page 1] ...e resolution due to averaging attention-weighted positions, an effect we counter...


In [10]:
# ## Part 9: Building a Document Navigator
# We create a helper class that wraps all retrieval functions into a
# single interface. This is the "vectorless database" for our paper.

class PageNavigator:
    """Navigate a document by page numbers and section names.

    No vectors, no embeddings. Pure structural retrieval.
    """

    def __init__(self, chunk_list):
        self.chunks = chunk_list
        self.sections = sorted(set(c["section"] for c in chunk_list))
        self.pages = sorted(set(c["page"] for c in chunk_list))

    def by_page(self, page_num):
        return retrieve_by_page(page_num, self.chunks)

    def by_section(self, section_name):
        return retrieve_by_section(section_name, self.chunks)

    def by_keyword(self, keyword):
        return [c for c in self.chunks if keyword.lower() in c["text"].lower()]

    def by_keyword_in_section(self, keyword, section_name):
        return retrieve_by_keyword_in_section(keyword, section_name, self.chunks)

    def by_page_range(self, start, end):
        return retrieve_by_page_range(start, end, self.chunks)

    def stats(self):
        return {
            "total_chunks": len(self.chunks),
            "total_pages": len(self.pages),
            "sections": self.sections,
        }

nav = PageNavigator(chunks)
info = nav.stats()
print(f"Navigator ready:")
print(f"  Chunks: {info['total_chunks']}")
print(f"  Pages: {info['total_pages']}")
print(f"  Sections: {info['sections']}")

Navigator ready:
  Chunks: 85
  Pages: 15
  Sections: ['3 Model Architecture', '4 Why Self-Attention', '5 Training', '6 Results', '7 Conclusion', 'Abstract']


In [11]:
# ## Part 10: Navigation Demo -- Answering Questions by Page
# We simulate queries and show how page-based navigation finds the
# right passages without any vector search.

demo_queries = [
    {"question": "What is the Transformer architecture?",
     "hint_page": 2,
     "hint_section": "Model Architecture"},
    {"question": "How is the model trained?",
     "hint_page": 6,
     "hint_section": "Training"},
    {"question": "What results did they achieve?",
     "hint_page": 7,
     "hint_section": "Results"},
]

for q in demo_queries:
    print(f"{'=' * 60}")
    print(f"Q: {q['question']}")
    print(f"  Strategy: navigate to page {q['hint_page']} ({q['hint_section']})")

    # Retrieve by section.
    results = nav.by_section(q["hint_section"])
    # Combine text for context.
    context = " ".join(c["text"][:200] for c in results[:2])
    preview = context[:200].replace("\n", " ")
    print(f"  Found {len(results)} chunks")
    print(f"  Preview: {preview}...")
    print()

Q: What is the Transformer architecture?
  Strategy: navigate to page 2 (Model Architecture)
  Found 25 chunks
  Preview: 1 Introduction Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks in particular, have been firmly established as state of the art approaches in sequence mod...

Q: How is the model trained?
  Strategy: navigate to page 6 (Training)
  Found 7 chunks
  Preview: length n is smaller than the representation dimensionality d, which is most often the case with sentence representations used by state-of-the-art models in machine translations, such as word-piece [38...

Q: What results did they achieve?
  Strategy: navigate to page 7 (Results)
  Found 13 chunks
  Preview: Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the English-to-German and English-to-French newstest2014 tests at a fraction of the training cost. Model B...



In [12]:
# ## Part 11: Highlighting the Keyword in Context
# For a given keyword, show exactly where it appears across the
# document with surrounding context.

def highlight_keyword(keyword, chunk_list, context_len=40):
    """Show every occurrence of a keyword with surrounding context."""
    keyword_lower = keyword.lower()
    occurrences = []
    for c in chunk_list:
        text_lower = c["text"].lower()
        idx = 0
        while True:
            idx = text_lower.find(keyword_lower, idx)
            if idx == -1:
                break
            start = max(0, idx - context_len)
            end = min(len(c["text"]), idx + len(keyword) + context_len)
            snippet = c["text"][start:end].replace("\n", " ")
            occurrences.append({
                "page": c["page"],
                "section": c["section"],
                "snippet": snippet,
            })
            idx += 1
    return occurrences

# Find all mentions of "self-attention" across the paper.
occurrences = highlight_keyword("self-attention", chunks)
print(f"Found {len(occurrences)} occurrences of 'self-attention':")
for i, occ in enumerate(occurrences[:6], 1):
    print(f"  {i}. [Page {occ['page']}, {occ['section']}]")
    print(f"     ...{occ['snippet']}...")
print(f"  ({len(occurrences) - 6} more)" if len(occurrences) > 6 else "")

Found 28 occurrences of 'self-attention':
  1. [Page 0, Abstract]
     ...dom. Jakob proposed replacing RNNs with self-attention and started the effort to evaluate this...
  2. [Page 1, 3 Model Architecture]
     ... Attention as described in section 3.2. Self-attention, sometimes called intra-attention is an...
  3. [Page 1, 3 Model Architecture]
     ...mpute a representation of the sequence. Self-attention has been used successfully in a variety...
  4. [Page 1, 3 Model Architecture]
     ... transduction model relying entirely on self-attention to compute representations of its input...
  5. [Page 1, 3 Model Architecture]
     ...will describe the Transformer, motivate self-attention and discuss its advantages over models ...
  6. [Page 2, 3 Model Architecture]
     ...this overall architecture using stacked self-attention and point-wise, fully connected layers ...
  (22 more)


In [13]:
# ## Part 12: Summary
# Page and section indexing gives us fast, deterministic retrieval
# with zero embeddings. The key advantages are:
#
# 1. Zero latency: no neural network calls needed.
# 2. Deterministic: same query always returns the same results.
# 3. Interpretable: you know exactly why a passage was retrieved.
# 4. Metadata-rich: page, section, and position are all available.
#
# Limitations:
# - Requires prior knowledge of document structure.
# - Cannot handle semantic queries like "What is similar to X?".
# - Works best with well-structured documents (papers, books, manuals).

print("Vectorless Page/Section Retrieval Summary:")
print("  Extract text and page numbers from PDF")
print("  Detect section headings with regex patterns")
print("  Assign sections to pages via forward-fill")
print("  Chunk text and attach page + section metadata")
print("  Retrieve by page number, section name, or keyword")
print("  No embeddings, no vectors, no latency")
print()
print("Next: Structured Metadata Filtering")

Vectorless Page/Section Retrieval Summary:
  Extract text and page numbers from PDF
  Detect section headings with regex patterns
  Assign sections to pages via forward-fill
  Chunk text and attach page + section metadata
  Retrieve by page number, section name, or keyword
  No embeddings, no vectors, no latency

Next: Structured Metadata Filtering
